# Local Projections Difference-in-Differences (LPDiD)

This tutorial introduces `LPDiD`, the Local Projections Difference-in-Differences estimator for the absorbing-treatment main path of Stata's `lpdid` command.

We will:

- load the official absorbing-treatment example data bundled in the test fixtures
- estimate default event-study and pooled effects
- compare `rw` and `nocomp` option paths
- draw an event-study plot from the `LPDiDResults` object
- show the corresponding Stata commands for parity checks


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from diff_diff import LPDiD


## 1. Load the Official LPDiD Example Data

The repository ships a trimmed CSV version of the official Stata `lpdidtestdata1.dta` example used for parity tests.


In [ ]:
df = pd.read_csv("tests/data/lpdidtestdata1_core.csv")
df.head()


## 2. Baseline LPDiD Fit

This reproduces the default absorbing-treatment event study and pooled pre/post effects.


In [ ]:
res = LPDiD(pre_window=5, post_window=10).fit(
    df,
    outcome="Y",
    unit="unit",
    time="time",
    treatment="treat",
)

res.event_study.head(12)


In [ ]:
res.pooled


## 3. Reweighting and Common-Composition Samples

`LPDiD` also supports the absorbing-treatment `rw` and `nocomp` options.


In [ ]:
res_rw = LPDiD(pre_window=5, post_window=10, reweight=True).fit(
    df,
    outcome="Y",
    unit="unit",
    time="time",
    treatment="treat",
)

res_nocomp = LPDiD(pre_window=5, post_window=10, no_composition=True).fit(
    df,
    outcome="Y",
    unit="unit",
    time="time",
    treatment="treat",
)

res_rw_nocomp = LPDiD(
    pre_window=5,
    post_window=10,
    reweight=True,
    no_composition=True,
).fit(
    df,
    outcome="Y",
    unit="unit",
    time="time",
    treatment="treat",
)

pd.concat(
    [
        res.pooled.assign(spec="baseline"),
        res_rw.pooled.assign(spec="rw"),
        res_nocomp.pooled.assign(spec="nocomp"),
        res_rw_nocomp.pooled.assign(spec="rw+nocomp"),
    ],
    ignore_index=True,
)[["spec", "window", "coefficient", "se", "n_obs"]]


## 4. Event-Study Plot

The `-1` period is the reference period, so we keep it on the line at zero but only draw confidence intervals for the estimated horizons.


In [ ]:
event = res_rw.event_study.copy().sort_values("horizon")
line_df = event.copy()
err_df = event[event["horizon"] != -1].copy()

plt.figure(figsize=(9, 5))
plt.plot(line_df["horizon"], line_df["coefficient"], marker="o", linestyle="-", label="LPDiD rw")
plt.errorbar(
    err_df["horizon"],
    err_df["coefficient"],
    yerr=[
        err_df["coefficient"] - err_df["conf_low"],
        err_df["conf_high"] - err_df["coefficient"],
    ],
    fmt="none",
    capsize=4,
    color="C0",
)
plt.axhline(0, color="black", linestyle="--", linewidth=1)
plt.axvline(-1, color="gray", linestyle=":", linewidth=1)
plt.xlabel("Event Time")
plt.ylabel("Treatment Effect")
plt.title("LPDiD Event Study (rw)")
plt.legend()
plt.tight_layout()
plt.show()


## 5. Stata Parity Commands

The corresponding Stata commands are:

```stata
use http://fmwww.bc.edu/repec/bocode/l/lpdidtestdata1.dta, clear

* Baseline
lpdid Y, unit(unit) time(time) treat(treat) pre(5) post(10) nograph

* Reweighted
lpdid Y, unit(unit) time(time) treat(treat) pre(5) post(10) rw nograph

* Common composition
lpdid Y, unit(unit) time(time) treat(treat) pre(5) post(10) nocomp nograph

* Reweighted + common composition
lpdid Y, unit(unit) time(time) treat(treat) pre(5) post(10) rw nocomp nograph
```

The test suite validates these absorbing-treatment paths against the Python implementation.
